## 2D materials: $\mathrm{MoS}_2$

Today we will analyse the electronic structure of monolayer $\mathrm{MoS_2}$, a 2D transition metal dichalogenide. Monolayer $\mathrm{MoS_2}$ has a direct band gap of around 1.8 eV, making it an excellent material for use in optoelectronics.

### 1.1. Setup

1. We will begin by finding the multilayer on https://next-gen.materialsproject.org. Click on  ```Start Exploring Materials``` and find $\mathrm{MoS_2}$ using the periodic system tool. 

2. Materials Project is really cool – do take a look around at which properties are available for $\mathrm{MoS_2}$. Then, download its structure in ```POSCAR``` format.

3. Open the ```POSCAR``` using Vesta (you might need to rename it to ```POSCAR```) and use the ```Objects -> Boundary``` option to view a supercell. How is it layered? Then, export a new ```POSCAR``` in Cartesian coordinates. 

4. Open the Cartesian ```POSCAR``` using your favourite text editor and edit it to keep only a single layer, with about 10 $\mathrm{\AA}$ of vacuum in the z direction to avoid self-interaction. Use Vesta to verify that you have the correct structure. 

5. Paste the atomic positions and cell parameters into the ```scf.in.template``` file. 

6. Download the PBE PAW JTH 1.1 pseudopotentials for Mo and S from https://www.pseudo-dojo.org/index.html. Note the recommended cutoffs in the lower left corner. Fill in the masses as well (NIST atomic spectroscopy data is a good resource). 

7. Execute the code below to load some common functions that we will use today. 

In [ ]:
"""
Remember, if you change something here you 
need to restart the kernel and execute
this cell again to import the changes. 
"""
from qe_funs import *

8. Now, inspect ```test_rho.sh``` and execute it. Plot the energy convergence with respect to the planewave cutoff energy. 

In [ ]:
!bash test_rho.sh

9. Now modify ```test_rho.sh``` and test for the convergence of energy with respect to the $\bf{k}$-grid. Which converges faster? What do you think is more important? 

In [ ]:
!bash test_k.sh # this command will better if you create test_k.sh

10. Ok, we have our parameters sorted. Now let's run a production calculation. 

In [ ]:
!pw.x < scf.in > scf.out

11. Create ```nscf_dos.in``` by copying ```scf.in```, crank up the $\bf{k}$-grid further, and run it. Then plot the density of states.

In [ ]:
!pw.x < nscf_dos.in > nscf_dos.out
# !dos.x < dos.in > dos.out
# e_range = 3.0
# plot_dos(e_range)

### 1.2 Projected density of states

The figure above is a bit dull, isn't it? Let's improve it! Inspect ```projwfc.in``` – it looks just like a ```dos.in```, but it also computes $\langle \phi_{i,l,m} | \psi_{n,\bf{k}} \rangle$, i.e. it projects each one-electron state $n$ onto atomic orbitals $\phi_{i,l,m}$ centred on atom $i$ with angular momentum $l$ and magnetic quantum number $m$. This enables us to plot the atom- and orbital-projected DOS, which tells us the character of the bands at each energy. 

12. To start, execute ```projwfc.in```.

In [ ]:
!projwfc.x < projwfc.in > projwfc.out

Wow, this produced many files. Let us first analyse ```projwfc.out```, focussing on the Löwdin charges at the bottom. How do they compare to your expected pattern of crystal field splitting produced by a trigonal prism?

 Now let's try plotting the projected DOS. To start, it might be useful to paste the ```plot_dos``` function from ```qe_funs.py```. Then, modify the code to plot the projected DOS in addition to the total DOS. 

13. What is the character of the bands close to the Fermi energy in $\mathrm{MoS_2}$? 

14. Is the total DOS a sum of all partial DOSes? 

### 1.3 Computing the band structure

Plotting the band structure in 1D wasn't too bad as $\bf{k}$ was really a $k$. In 2D and 3D, $\bf{k}$ is a vector and we need to plot the band structure along the high-symmetry lines, which capture the interesting features in the band structure. Fortunately, the SeekPath tool can help.

15. Navigate to https://seekpath.materialscloud.io and feed it your ```scf.in```. Determine the high-symmetry lines, keeping in mind that you are interested in the monolayer, and there is no spin-orbit coupling.  

16. Create a ```nscf_bands.in``` using the following syntax:
```
K_POINTS crystal_b
m
kx_1 ky_1 kz_1 n
kx_2 ky_2 kz_2 n
...
kx_m ky_m kz_m 0 
```
This will create a path connecting $m$ high-symmetry points, with $n$ points between each two points. For example, in the case of a 1D polymer you had:
```
K_POINTS crystal_b
2
0.0 0.0 0.0 50 ! gamma
0.5 0.0 0.0 0  ! X
```

17. Execute ```nscf_bands.in``` with $n = $ 25–35 and post-process by running ```bands.in```.

In [ ]:
!pw.x < nscf_bands.in > nscf_bands.out
!bands.x < bands.in > bands.out

### 1.4 Plotting the band structure

18. Use the code below to plot the band structure. Note that we need to define the ticks somewhat manually. 

In [ ]:
file_name = "bands.dat.gnu"
e_fermi = 2.5325
e_range = 5.0 
n_bands = 17

bands = np.genfromtxt(file_name) 
bands = np.split(bands, n_bands) 

for band in bands:
    x = band[:, 0]
    y = band[:, 1]
    if np.all(y > e_fermi):
        plt.plot(x, y, color=colours["orange"])
    else:
        plt.plot(x, y, color=colours["blue"])
plt.ylim(e_fermi - e_range, e_fermi + e_range)
plt.xlim(0, np.max(bands[0][:, 0]))
plt.xlabel(r"$\bf{k}$")
# define the ticks, a bit tedious
plt.xticks([(bands[0][0, 0]), (bands[0][25, 0]), (bands[0][50, 0]), (bands[0][75, 0])], [r"$\Gamma$", r"M", r"K", r"$\Gamma$"])
plt.axvline(bands[0][25, 0], linestyle='dashed', color = "k") # M point
plt.axvline(bands[0][50, 0], linestyle='dashed', color = "k") # K point
plt.ylabel("energy (eV)")
plt.show()

### 1.5 Projected band structure (optional)

If you want to be _really fancy_, you can use ```projwfc.x``` to determine the band character in terms of contributing orbitals or atoms. This can then be plotted as a band colour, see e.g. https://pubs.acs.org/doi/10.1021/jacs.3c12079 or Fig. 1 in https://onlinelibrary.wiley.com/doi/full/10.1002/anie.202307035 for examples. 

19. We will not go through plotting projected band structures, but you are welcome to give it a go. The command you need to get started is below.  

In [ ]:
!projwfc.x < projwfc_bands.in > projwfc.out

## 2. Effective mass

The conductivity of the free electron gas in the Drude model is given by: 

$$\sigma = \frac{ne^2\tau}{m_\mathrm{e}}$$
 
where $n$ is the electron density, $e$ the elementary charge, $\tau$  the scattering time, and ${m_\mathrm{e}}$ the electron mass. We can extend this formula to estimate conductivity in materials, replacing  ${m_\mathrm{e}}$ with the **effective mass** $m^*$. The effective mass reflects interactions between the charge carrier, which may be either the  electron or the hole, with the periodic lattice, while $\tau$ captures scattering from defects, phonons, and other imperfections. 

The effective mass arises from the curvature of the energy bands. Near a band extremum (minimum for electrons, maximum for holes), we can approximate the dispersion relation as parabolic:

$$E(\mathbf{k}) \approx E_0 + \frac{\hbar^2}{2m^*} (\mathbf{k} - \mathbf{k}_0)^2$$

From this, we can extract the effective mass from the band curvature:

$$m^* = \hbar^2 \left( \frac{d^2E}{dk^2} \right)^{-1}$$

While the formula above is valid for 1D systems, in general the effective mass is a **tensor** because the band curvature can be different in different crystal directions:

$$\frac{1}{m^*_{ij}} = \frac{1}{\hbar^2}\frac{\partial^2 E}{\partial k_i \partial k_j}$$

or in matrix form:

$$\mathbf{m^*} = \begin{pmatrix}
m^*_{xx} & m^*_{xy} & m^*_{xz} \\
m^*_{yx} & m^*_{yy} & m^*_{yz} \\
m^*_{zx} & m^*_{zy} & m^*_{zz}
\end{pmatrix}$$

The isotropic macroscopic effective mass is a harmonic mean of its principal components: 

$$m^* = 3\left(\frac{1}{m^*_{xx}} + \frac{1}{m^*_{yy}} + \frac{1}{m^*_{zz}}\right)^{-1}$$

In orthorhombic systems we can, at least in principle, macroscopically differentiate between $m^*_{xx}$, $m^*_{yy}$, and $m^*_{zz}$. In hexagonal materials such as $\mathrm{MoS_2}$, macroscopically we can differentiate between the in-plane:

$$m^*_{\parallel} = 2\left(\frac{1}{m^*_{xx}} + \frac{1}{m^*_{yy}}\right)^{-1}$$

and out-of-plane effective masses:

$$m^*_{\perp} = m^*_{zz}$$

Effective masses are important because they provide a link between a computable microscopic quantity (band curvature) and a measurable macroscopic quantity (conductivity). They are typically computed at the conduction band minimum (CBM) and valence band maximum (VBM), where electrons and holes are charge carriers, respectively. These points are most easily accessible by doping and thermal or optical excitation.


20. Now go back to the band structure above. Where are the CBM and VBM?

We will cheat a little: experiments show that monolayer $\mathrm{MoS_2}$ is a direct-gap semiconductor (https://journals.aps.org/prl/abstract/10.1103/PhysRevLett.105.136805, 13k citations!), but that $\Gamma$ and M are near-degenerate in the VB. This paper experimentally estimates effective masses at K at $m^* = 0.4$ for the electron and at $m^* = 0.6$ for the hole. 

21. Think carefully how you could determine the effective mass accurately from DFT. 

Hint #1: once you have the SCF converged, you can sample any section of the $\bf{k}$-space you want. 

Hint #2: there is a function for determining effective mass in ```qe_funs.py```.

## 3. Beyond GGA (optional)

22. The experimental band gap of monolayer $\mathrm{MoS_2}$ is 1.8–1.9 eV. What value did you get? 

While for $\mathrm{MoS_2}$ PBE isn't too bad, its performance can typically be improved by reducing its self-interaction error. As we learnt last week, GGA functionals over-hybridise the orbitals and over-delocalise the electrons. This is usually addressed by adding exact exchange either empirically (hybrid DFT) or rigorously (GW). A cheaper alternative is DFT+U. 

### DFT+U

Conceptually, DFT+U adds Hubbard-like term to the Hamiltonian: 

$$H = H_{\mathrm{DFT}} + \sum_{i,l,s} U_{i,l} \ n_{i,l,s} (1 - n_{i,l,s})$$

where $U_{i,l}$ is the energy added according to the occupations of atomic orbital $l$ on atom $i$, which are computed as in the ```projwfc.in``` routine, and $s$ runs over spins. This correction pushes the orbitals $\phi_{i,l}$ towards occupations of 0 or 1, potentially decreasing their hybridisation. 

The value of $U$ is usually determined empirically, or used as a parameter to explore the possible behaviour. DFT+U is popular because it adds very little computational overhead. 

23. Here is a minimal input for $\mathrm{MoS_2}$, adding $U$ as well as $V$, which corresponds to nearest-neighbour Coulombic repulsion between orbitals on Mo and S. What do you get by including this correction?
```
HUBBARD (ortho-atomic)
U Mo-4d 2.5 
V Mo-4d S-3p 1 2 2.0 ! inter-site repulsion between atoms 1 and 2
V Mo-4d S-3p 1 3 2.0 ! inter-site repulsion between atoms 1 and 3
```